# ONIA · Regularizarea L1 și L2

Underfitting, overfitting, validare, Ridge, Lasso și Elastic Net. Datele din exemple sunt pentru demonstrație.

## Separare și validare cu scikit-learn

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

X, y = load_diabetes(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
scoruri = cross_val_score(model, X_dev, y_dev, cv=5,
                         scoring="neg_mean_squared_error")
print("MSE mediu la validare:", round(-scoruri.mean(), 2))
# Testul separat se consultă abia după alegerea modelului.


## Gradient descent cu penalizare L2

In [ ]:
def antreneaza_ridge(X, y, lam=0.1, rata=0.03, epoci=2000):
    n, k = len(X), len(X[0])
    w = [0.0] * k
    b = 0.0
    for _ in range(epoci):
        gw = [0.0] * k
        gb = 0.0
        for i in range(n):
            predictie = b
            for j in range(k):
                predictie += w[j] * X[i][j]
            eroare = predictie - y[i]
            gb += eroare / n
            for j in range(k):
                gw[j] += eroare * X[i][j] / n
        for j in range(k):
            w[j] -= rata * (gw[j] + lam * w[j])
        b -= rata * gb
    return w, b

X = [[1], [2], [3], [4]]
y = [2, 3, 3, 5]
w, b = antreneaza_ridge(X, y, lam=0.15)
print(round(w[0], 3), round(b, 3))


## Lasso proximal din liste Python

In [ ]:
def soft_threshold(v, prag):
    if v > prag:
        return v - prag
    if v < -prag:
        return v + prag
    return 0.0

def antreneaza_lasso(X, y, lam=0.10, rata=0.01, epoci=4000):
    n, k = len(X), len(X[0])
    w = [0.0] * k
    b = sum(y) / n
    for _ in range(epoci):
        gw = [0.0] * k
        gb = 0.0
        for i in range(n):
            p = b + sum(w[j] * X[i][j] for j in range(k))
            e = p - y[i]
            gb += e / n
            for j in range(k):
                gw[j] += e * X[i][j] / n
        for j in range(k):
            u = w[j] - rata * gw[j]
            w[j] = soft_threshold(u, rata * lam)
        b -= rata * gb
    return w, b

X = [[-2, 1], [-1, -1], [0, 1],
     [1, -1], [2, 1], [3, -1]]
y = [-3, -1, 1, 3, 5, 7]
w, b = antreneaza_lasso(X, y, lam=0.2)
print([round(v, 3) for v in w], round(b, 3))


## Pipeline corect: scalare urmată de Lasso

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.1, max_iter=20000)
)
model.fit([[1, 1000], [2, 1200], [3, 1300],
           [4, 1500], [5, 1800]], [3, 5, 5, 7, 8])
print(model.predict([[3.5, 1400]]).round(2))


## Comparație practică a patru regresori

In [ ]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error

X, y = make_regression(n_samples=220, n_features=12,
                       n_informative=3, noise=25, random_state=7)
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=0.3, random_state=12
)
modele = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=10),
    "Lasso": Lasso(alpha=2, max_iter=20000),
    "ElasticNet": ElasticNet(alpha=1, l1_ratio=0.5,
                             max_iter=20000)
}
for nume, estimator in modele.items():
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(Xtr, ytr)
    pred = model.predict(Xva)
    print(nume, "MSE:", round(mean_squared_error(yva, pred), 2))


## Clasificator logistic cu regularizare și scalare

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.1, solver="lbfgs", max_iter=3000)
)
model.fit(Xtr, ytr)
pred = model.predict(Xva)
print(round(accuracy_score(yva, pred), 3))
print(round(f1_score(yva, pred), 3))


## PolynomialFeatures + Ridge într-un singur Pipeline

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge

X = np.linspace(-2, 2, 35).reshape(-1, 1)
y = 0.3 * X[:, 0] ** 2 + np.sin(X[:, 0])
model = make_pipeline(
    PolynomialFeatures(degree=5, include_bias=False),
    StandardScaler(),
    Ridge(alpha=1.5)
)
model.fit(X, y)
print(model.predict(np.array([[0.0], [1.0]])).round(3))


## GridSearchCV selectează intensitatea Ridge

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

X, y = load_diabetes(return_X_y=True)
Xtr, Xtest, ytr, ytest = train_test_split(
    X, y, test_size=0.2, random_state=42
)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])
cautare = GridSearchCV(
    pipe, {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring="neg_mean_squared_error"
)
cautare.fit(Xtr, ytr)
print("Alpha ales:", cautare.best_params_["model__alpha"])
print("MSE cross-validation:", round(-cautare.best_score_, 2))
# Xtest rămâne pentru verificarea finală.


## Clasificare logistică regularizată — fără biblioteci ML

In [ ]:
import math

def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    t = math.exp(z)
    return t / (1 + t)

def logistic_l2(x, y, lam=0.2, rata=0.2, epoci=1000):
    w = 0.0
    b = 0.0
    n = len(x)
    for _ in range(epoci):
        gw = 0.0
        gb = 0.0
        for i in range(n):
            p = sigmoid(w * x[i] + b)
            err = p - y[i]
            gw += err * x[i] / n
            gb += err / n
        w -= rata * (gw + lam * w)
        b -= rata * gb
    return w, b

x = [-2, -1, -0.5, 0, 1, 2, 3]
y = [0, 0, 0, 0, 1, 1, 1]
w, b = logistic_l2(x, y)
print(round(w, 3), round(b, 3))
print(round(sigmoid(0.6 * w + b), 3))


## Early stopping simplu pentru regresie

In [ ]:
xtrain, ytrain = [1, 2, 3, 4], [2, 3, 5, 4]
xval, yval = [1.5, 3.5], [2.7, 4.3]

w, b = 0.0, 0.0
cel_mai_bun = float("inf")
parametri = (w, b)
rabdare = 80
fara_progres = 0

for _ in range(3000):
    n = len(xtrain)
    gw = sum((w * xtrain[i] + b - ytrain[i]) *
             xtrain[i] for i in range(n)) / n
    gb = sum(w * xtrain[i] + b - ytrain[i]
             for i in range(n)) / n
    w -= 0.02 * gw
    b -= 0.02 * gb
    mse_val = sum((w * xval[i] + b - yval[i]) ** 2
                  for i in range(len(xval))) / len(xval)
    if mse_val < cel_mai_bun - 1e-8:
        cel_mai_bun = mse_val
        parametri = (w, b)
        fara_progres = 0
    else:
        fara_progres += 1
    if fara_progres >= rabdare:
        break

print(tuple(round(t, 3) for t in parametri))
